In [1]:
# This part helps with stability on multi-GPU systems with great inbalance between the GPUs (eg. integrated vs discrete gpu)
# IMPORTANT must be done before importing torch, else session must be restarted
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch

for i in range(torch.cuda.device_count()):
    free = torch.cuda.mem_get_info(i)[0]
    total = torch.cuda.mem_get_info(i)[1]

    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Free:  {free / 1024**3:.2f} GB")
    print(f"  Total: {total / 1024**3:.2f} GB")

GPU 0: AMD Radeon RX 6700S
  Free:  7.96 GB
  Total: 7.98 GB


## Load Dataset

This portion loads dataset, and assigns id for each label

In [2]:
from util import load_datasets

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets(
    train_path="./../datasets/spam-messages/train/data-00000-of-00001.arrow",
    val_path="./../datasets/spam-messages/validation/data-00000-of-00001.arrow",
    test_path="./../datasets/spam-messages/test/data-00000-of-00001.arrow",
    label2id=label2id
)

Loaded datasets (train=47392, val=5923, test=5926)


Map:   0%|          | 0/47392 [00:00<?, ? examples/s]

Map:   0%|          | 0/5923 [00:00<?, ? examples/s]

Map:   0%|          | 0/5926 [00:00<?, ? examples/s]

In [2]:
from util import clear_cache

clear_cache("./../datasets/spam-messages/train")
clear_cache("./../datasets/spam-messages/validation")
clear_cache("./../datasets/spam-messages/test")

## Model settings 

### XLM-RoBERTA

In [ ]:
from util import tokenize_dataset, load_model_and_tokenizer, build_trainer


model_name = "FacebookAI/xlm-roberta-base"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset = tokenize_dataset(train_dataset, tokenizer)
val_dataset = tokenize_dataset(val_dataset, tokenizer)
test_dataset = tokenize_dataset(test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/47392 [00:00<?, ? examples/s]

Map:   0%|          | 0/5923 [00:00<?, ? examples/s]

Map:   0%|          | 0/5926 [00:00<?, ? examples/s]

### TinyBert

In [3]:
from util import tokenize_dataset, load_model_and_tokenizer, build_trainer


model_name = "huawei-noah/TinyBERT_General_4L_312D"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset = tokenize_dataset(train_dataset, tokenizer)
val_dataset = tokenize_dataset(val_dataset, tokenizer)
test_dataset = tokenize_dataset(test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

Map:   0%|          | 0/47392 [00:00<?, ? examples/s]

Map:   0%|          | 0/5923 [00:00<?, ? examples/s]

Map:   0%|          | 0/5926 [00:00<?, ? examples/s]

## Train

In [7]:
trainer.train()
trainer.save_model("my-model")
tokenizer.save_pretrained("my-model")

TypeError: '<=' not supported between instances of 'float' and 'Dataset'

In [ ]:
trainer.evaluate()

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="my-model")
classifier("This is amazing!")